# Hybrid search RAG in Python: BM25 + vector, fused in one SQL call

Keyword search (BM25) nails exact terms — error codes, IDs, product names — but misses paraphrases. Vector search captures meaning but can miss a literal token. **Hybrid search** combines both so you don't have to choose.

[Infino](https://pypi.org/project/infino/) fuses them **inside the engine**: one SQL `hybrid_search(...)` call runs BM25 and vector k-NN and merges them with Reciprocal Rank Fusion (RRF) — over a single copy of your data, no separate vector database and no client-side merge.

We measure it on **MS MARCO**, a standard IR benchmark with ground-truth relevance labels, reporting **recall@10** for BM25-only, vector-only, and hybrid on the same data.

## Setup

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import sys
from pathlib import Path

# Make the repo's shared helpers importable.
sys.path.insert(0, str(Path.cwd().parent))

import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.loaders import load_ms_marco
from _shared.sql import sql_lit, query

## 1. Load MS MARCO (with relevance labels)

Each query comes with candidate passages, one or more flagged relevant. We flatten the passages into a corpus (each with a stable `pid`) and keep the relevant `pid`s per query so we can score retrieval.

In [3]:
passages, queries = load_ms_marco(n_queries=200)
print(f"{len(passages)} passages, {len(queries)} labeled queries")
print("example query:", queries[0]["query"])

1743 passages, 200 labeled queries
example query: walgreens store sales average


## 2. Index the passages (local disk)

One table, two indexes: `text` for BM25 and `emb` for vectors — over the same rows. A `pid` column lets us match results back to the relevance labels.

In [4]:
DB_DIR = "./marco_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("pid", pa.int64(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = infino.IndexSpec().fts("text").vector("emb", DIM, n_cent=256, metric=METRIC)
table = db.create_table("passages", schema, spec)

embeddings = embed([p["text"] for p in passages])
table.append(pa.record_batch([
    pa.array([p["pid"] for p in passages], type=pa.int64()),
    pa.array([p["text"] for p in passages], type=pa.large_utf8()),
    as_vector_column(embeddings),
], schema=schema))
print("indexed", db.query_sql("SELECT COUNT(*) AS n FROM passages").to_pydict()["n"][0], "passages")

indexed 1743 passages


## 3. Three retrievers as SQL table functions

Infino exposes retrieval as SQL table functions you can compose. The key one is **`hybrid_search`** — a single call that fuses BM25 and vector k-NN with RRF inside the engine. Each function takes the table name first; we JOIN back to `passages` to recover the `pid`, ordered best-first.

In [5]:
K = 10


def retrieve(mode: str, q: str) -> list[int]:
    qvec = ",".join(str(x) for x in embed_query(q))
    if mode == "bm25":
        tvf = f"bm25_search('passages', 'text', {sql_lit(q)}, {K})"
        order = "DESC"  # higher BM25 score = better
    elif mode == "vector":
        tvf = f"vector_search('passages', 'emb', '{qvec}', {K})"
        order = "ASC"   # smaller distance = nearer
    else:  # hybrid — one call, fused in-engine
        tvf = f"hybrid_search('passages', 'text', {sql_lit(q)}, 'emb', '{qvec}', {K})"
        order = "DESC"  # higher fused (RRF) score = better
    sql = f"SELECT p.pid FROM {tvf} s JOIN passages p ON s._id = p._id ORDER BY s.score {order}"
    return query(db, sql).get("pid", [])

## 4. Measure recall@10 on real labels

Recall@10 = the fraction of queries whose relevant passage shows up in the top 10.

In [6]:
def recall_at_k(mode: str) -> float:
    hits = sum(
        bool(set(q["relevant_pids"]) & set(retrieve(mode, q["query"])))
        for q in queries
    )
    return hits / len(queries)


for mode in ("bm25", "vector", "hybrid"):
    print(f"recall@10  {mode:>6}: {recall_at_k(mode):.3f}")

recall@10    bm25: 0.895


recall@10  vector: 0.995


recall@10  hybrid: 0.995


On MS MARCO, dense vectors are already near-optimal and BM25 trails — so the headline here is that **hybrid matches the best single retriever**, never underperforming it. The win is larger on corpora where lexical and semantic signals diverge (exact codes, names, jargon mixed with prose): there BM25 and vector miss *different* queries, and RRF recovers both. The next cell shows one such query in this sample.

## 5. A concrete case: hybrid rescues a keyword query

Find a query whose relevant passage falls outside BM25's top 10 but inside hybrid's, and show where each retriever ranks it.

In [7]:
def rank_of_relevant(mode: str, q: dict):
    """1-based rank of the first relevant passage, or None if outside top-K."""
    relevant = set(q["relevant_pids"])
    for i, pid in enumerate(retrieve(mode, q["query"]), start=1):
        if pid in relevant:
            return i
    return None


rescued = next(
    q for q in queries
    if rank_of_relevant("bm25", q) is None and rank_of_relevant("hybrid", q) is not None
)
by_pid = {p["pid"]: p["text"] for p in passages}

print("query:", rescued["query"])
print(f"relevant passage rank  —  BM25: outside top {K}   hybrid: #{rank_of_relevant('hybrid', rescued)}")
print("\nthe relevant passage hybrid recovered:")
print("  ", by_pid[rescued["relevant_pids"][0]][:160])

query: how much do bartenders make
relevant passage rank  —  BM25: outside top 10   hybrid: #7

the relevant passage hybrid recovered:
   According to the Bureau of Labor Statistics, the average hourly wage for a bartender is $10.36, and the average yearly take-home is $21,550. Bartending can be a


## What just happened

`hybrid_search` ran BM25 and vector k-NN over **one copy of the data** and fused them with RRF in a single SQL call — no second system, no client-side merge. Because it is plain SQL, the result set can also feed joins, filters, and aggregates in the same query.

**Next:** [`03_filtered_rag.ipynb`](03_filtered_rag.ipynb) adds metadata filters (price, category) to retrieval — multi-tenant search over one table.